In [85]:
import numpy as np
import torch
import torch.nn as nn
import torchmetrics
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Pytorch Fundamentals

In [2]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]],
                 dtype=torch.float16)
X

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float16)

In [3]:
X.shape

torch.Size([2, 3])

In [4]:
X.dtype

torch.float16

In [5]:
X[0, 1]

tensor(4., dtype=torch.float16)

In [6]:
X[:, 1]

tensor([4., 3.], dtype=torch.float16)

In [7]:
# torch.abs(), torch.cos(), torch.exp(), torch.max(), torch.mean(), torch.sqrt()
print(f"exponentiation: {X.exp()}")
print(f"mean: {X.mean()}")
print(f"max values per columns: {X.max(dim=0)}")

exponentiation: tensor([[   2.7188,   54.5938, 1097.0000],
        [   7.3906,   20.0781,  403.5000]], dtype=torch.float16)
mean: 3.833984375
max values per columns: torch.return_types.max(
values=tensor([2., 4., 7.], dtype=torch.float16),
indices=tensor([1, 0, 0]))


In [8]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]], dtype=torch.float16)

In [9]:
# matrix tranpose and matrix multiplication
X @ X.T

tensor([[66., 56.],
        [56., 49.]], dtype=torch.float16)

In [10]:
# biz dim argumenti-nin yerinə axis də yaza bilərik
X.max(axis=0)

torch.return_types.max(
values=tensor([2., 4., 7.], dtype=torch.float16),
indices=tensor([1, 0, 0]))

In [11]:
X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float16)

In [12]:
torch.tensor(np.array([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [14]:
torch.FloatTensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [15]:
X[:, 1] = -99
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]], dtype=torch.float16)

In [16]:
X.relu_()
X

tensor([[1., 0., 7.],
        [2., 0., 6.]], dtype=torch.float16)

In [17]:
X[:, 1] = -99
X.abs_()
X

tensor([[ 1., 99.,  7.],
        [ 2., 99.,  6.]], dtype=torch.float16)

In [18]:
X.sqrt_()
X

tensor([[1.0000, 9.9531, 2.6465],
        [1.4141, 9.9531, 2.4492]], dtype=torch.float16)

In [19]:
X.zero_()
X

tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float16)

In [20]:
# hardware acceleration
if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
else:
  device = "cpu"

In [21]:
# create a tensor on the CPU
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

# copy it to the GPU
M = M.to(device)

In [22]:
# M.cuda(), M.cpu()

In [23]:
M.device

device(type='cuda', index=0)

In [24]:
# creating a tensor on the GPU
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)
M

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

In [25]:
# comparing the the speed of matrix multiplication running on the CPU versus on the GPU
M = torch.rand((1000, 1000))
%timeit M @ M.T

M = torch.rand((1000, 1000), device="cuda")
%timeit M @ M.T

16.9 ms ± 2.45 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
572 µs ± 3.88 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [26]:
# Autograd -> hər bir neyronun gradientini avtomatik olaraq hesablayır
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [27]:
f.backward()
x.grad

tensor(10.)

In [28]:
learning_rate = 0.1
with torch.no_grad():
  x -= learning_rate * x.grad

In [29]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

In [30]:
x.grad.zero_()

tensor(0.)

In [31]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
  f = x ** 2 # forward pass
  f.backward() # backward pass
  with torch.no_grad():
    x -= learning_rate * x.grad # gradient descent step
  x.grad.zero_() # reset the gradients

# Implementing a Linear Regression

In [32]:
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, test_size=0.2, random_state=42)

In [33]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

In [34]:
X_train.shape, X_valid.shape, X_test.shape

((13209, 8), (3303, 8), (4128, 8))

In [35]:
y_train.shape, y_valid.shape, y_test.shape

((13209,), (3303,), (4128,))

In [36]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

In [37]:
X_train.shape, X_valid.shape, X_test.shape

(torch.Size([13209, 8]), torch.Size([3303, 8]), torch.Size([4128, 8]))

In [38]:
means = X_train.mean(dim=0, keepdims=True)
std = X_train.std(dim=0, keepdims=True)

X_train = (X_train - means) / std
X_valid = (X_valid - means) / std
X_test = (X_test - means) / std

In [39]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [41]:
# creating the parameters of the linear regression model
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [42]:
learning_rate=0.4
n_epochs = 20
for epoch in range(n_epochs):
  y_pred = X_train @ w + b
  loss = ((y_pred - y_train) ** 2).mean()
  loss.backward()
  with torch.no_grad():
    b -= learning_rate * b.grad
    w -= learning_rate * w.grad
    b.grad.zero_()
    w.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, loss: {loss.item()}")

Epoch 1/20, loss: 16.0495548248291
Epoch 2/20, loss: 4.726516246795654
Epoch 3/20, loss: 2.14831280708313
Epoch 4/20, loss: 1.26175057888031
Epoch 5/20, loss: 0.9224843978881836
Epoch 6/20, loss: 0.7815981507301331
Epoch 7/20, loss: 0.7158308029174805
Epoch 8/20, loss: 0.6797963380813599
Epoch 9/20, loss: 0.6563625931739807
Epoch 10/20, loss: 0.6388930082321167
Epoch 11/20, loss: 0.624715268611908
Epoch 12/20, loss: 0.6126787066459656
Epoch 13/20, loss: 0.6022298336029053
Epoch 14/20, loss: 0.5930587649345398
Epoch 15/20, loss: 0.584962010383606
Epoch 16/20, loss: 0.577788770198822
Epoch 17/20, loss: 0.5714176297187805
Epoch 18/20, loss: 0.5657473802566528
Epoch 19/20, loss: 0.5606915950775146
Epoch 20/20, loss: 0.5561758875846863


In [43]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = X_new @ w + b

In [44]:
y_pred

tensor([[0.9126],
        [1.6173],
        [2.6462]])

In [45]:
# Linear Regression using Pytorch's High Level API
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [46]:
model.bias, model.weight

(Parameter containing:
 tensor([0.3117], requires_grad=True),
 Parameter containing:
 tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
        requires_grad=True))

In [47]:
model.parameters()

<generator object Module.parameters at 0x78d7dca31a80>

In [48]:
next(model.parameters())

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [49]:
for param in model.parameters():
  print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [50]:
model(X_train[:2])

tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

In [51]:
# creating an optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [54]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch  +1}/{n_epochs}, loss: {loss.item()}")

In [55]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, loss: 4.270204544067383
Epoch 2/20, loss: 0.7620968222618103
Epoch 3/20, loss: 0.6113183498382568
Epoch 4/20, loss: 0.5919578075408936
Epoch 5/20, loss: 0.581246018409729
Epoch 6/20, loss: 0.5727843046188354
Epoch 7/20, loss: 0.5656700730323792
Epoch 8/20, loss: 0.5595568418502808
Epoch 9/20, loss: 0.5542512536048889
Epoch 10/20, loss: 0.5496227145195007
Epoch 11/20, loss: 0.5455722808837891
Epoch 12/20, loss: 0.5420194864273071
Epoch 13/20, loss: 0.5388972759246826
Epoch 14/20, loss: 0.5361485481262207
Epoch 15/20, loss: 0.5337244868278503
Epoch 16/20, loss: 0.5315830111503601
Epoch 17/20, loss: 0.5296878814697266
Epoch 18/20, loss: 0.5280079245567322
Epoch 19/20, loss: 0.5265161991119385
Epoch 20/20, loss: 0.5251892805099487


In [56]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = model(X_new)

In [57]:
y_pred

tensor([[0.8241],
        [1.6844],
        [2.6655]])

# Implementing a Regression MLP

In [58]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [59]:
learning_rate - 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [60]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, loss: 4.971917629241943
Epoch 2/20, loss: 13.225018501281738
Epoch 3/20, loss: 11.682997703552246
Epoch 4/20, loss: 1.675215721130371
Epoch 5/20, loss: 1.2973839044570923
Epoch 6/20, loss: 1.2610008716583252
Epoch 7/20, loss: 1.2110871076583862
Epoch 8/20, loss: 1.1327025890350342
Epoch 9/20, loss: 1.0145070552825928
Epoch 10/20, loss: 0.876427412033081
Epoch 11/20, loss: 0.7812116742134094
Epoch 12/20, loss: 0.7388125061988831
Epoch 13/20, loss: 0.7117041945457458
Epoch 14/20, loss: 0.6886736750602722
Epoch 15/20, loss: 0.6689361333847046
Epoch 16/20, loss: 0.6542356014251709
Epoch 17/20, loss: 0.6534022092819214
Epoch 18/20, loss: 0.7158243656158447
Epoch 19/20, loss: 0.9159135818481445
Epoch 20/20, loss: 1.7832328081130981


# Implementing Mini-Batch Gradient Descent using DataLoaders

In [62]:
train_dataset = TensorDataset(X_train, y_train)

In [64]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [65]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

model.to(device)

Sequential(
  (0): Linear(in_features=8, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=40, bias=True)
  (3): ReLU()
  (4): Linear(in_features=40, out_features=1, bias=True)
)

In [66]:
def train(model, optimizer, criterion, train_loader, n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [67]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 4.9720
Epoch 2/20, Loss: 4.9724
Epoch 3/20, Loss: 4.9719
Epoch 4/20, Loss: 4.9707
Epoch 5/20, Loss: 4.9713
Epoch 6/20, Loss: 4.9719
Epoch 7/20, Loss: 4.9708
Epoch 8/20, Loss: 4.9720
Epoch 9/20, Loss: 4.9736
Epoch 10/20, Loss: 4.9717
Epoch 11/20, Loss: 4.9708
Epoch 12/20, Loss: 4.9743
Epoch 13/20, Loss: 4.9714
Epoch 14/20, Loss: 4.9715
Epoch 15/20, Loss: 4.9711
Epoch 16/20, Loss: 4.9724
Epoch 17/20, Loss: 4.9723
Epoch 18/20, Loss: 4.9720
Epoch 19/20, Loss: 4.9716
Epoch 20/20, Loss: 4.9722


# Model Evaluation

In [68]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
  model.eval()
  metrics = []
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric = metric_fn(y_pred, y_batch)
      metrics.append(metric)
  return aggregate_fn(torch.stack(metrics))

In [69]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)

In [78]:
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(5.0886, device='cuda:0')

In [79]:
def rmse(y_pred, y_true):
  return ((y_pred - y_true) ** 2).mean().sqrt()

In [80]:
evaluate(model, valid_loader, rmse)

tensor(2.2425, device='cuda:0')

In [81]:
valid_mse.sqrt()

tensor(2.2558, device='cuda:0')

In [82]:
evaluate(model, valid_loader, mse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(2.2558, device='cuda:0')

In [84]:
#!pip install torchmetrics

In [86]:
def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)
  return metric.compute()

In [87]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

In [88]:
evaluate_tm(model, valid_loader, rmse)

tensor(2.2595, device='cuda:0')

# Building Nonsequential Models using Custom Modules